**Status: live in-race podium and winner chances (replaces `live_model` from `phase_3_ranking.ipynb`).**

The old lap-level ranker was no better than the running order (top-3 0.791 vs 0.799 on 2022-23). This notebook has two models, each judged against that baseline:

- **Podium (early-race hybrid):** up to 50% race distance, a shallow XGBoost on the pre-race podium probability (`f1_podium_prior_oof`) plus lap state (position, places lost or gained since the grid, gaps to P3/P4, pit stops still owed, recent pace, tyre age, safety car). After 50% it uses the running order, which every model lost to late in the race.
- **Winner:** a logistic model over the top 4 cars (pre-race probability, gaps, closing rate, pace and tyre-age difference to the car ahead, pit stops owed, laps left, safety car), scored against "the current leader wins".

Rules fixed in advance (design chosen on 2022-23, tested once on 2024-26): podium ships if the race-level 95% CI of the 0-50% top-3 gain over the running order is above zero (passed: +2.5 pts [+0.8, +4.3]); winner ships if the 70-100% gain CI is above zero (failed: +2.8 [-0.5, +6.8]; mid-race +4 to +5 pts was significant). The walk-forward backtest below re-measures both on every season from 2022 and stores the flags; the App shows the winner chance as experimental whatever the flag says.

Predictions are walk-forward by season (trained only on earlier seasons), from 2021. Writes `f1_live_predictions` and `f1_live_backtest`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

FIRST_TRAIN_YEAR = 2020
EARLY_CUT = 0.5
EARLY_TRAIN_MAX = 0.7
K = ["Year", "Race", "LapNumber"]
D = ["Year", "Race", "Driver"]

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from live.features import EARLY, build_features, fit_podium, fit_win, norm_to, running_lookup, win_x

cols = ["Year", "Race", "Driver", "LapNumber", "Position", "LapTime", "Time", "Stint", "TyreLife", "TrackStatus", "PitInTime", "PitOutTime"]
laps = spark.table("workspace.default.f1_master_lap_dataset").select(*cols).toPandas()
prior = spark.table("workspace.default.f1_podium_prior_oof").toPandas()
d = build_features(laps[laps["Year"] >= FIRST_TRAIN_YEAR], prior)
print("Races per season:", d.groupby("Year")["Race"].nunique().to_dict())

In [ ]:
out = []
for year in range(FIRST_TRAIN_YEAR + 1, int(d["Year"].max()) + 1):
    tr, te = d[d["Year"] < year], d[d["Year"] == year].copy()
    pod, win = fit_podium(tr), fit_win(tr)
    tab, bucket = running_lookup(tr)
    te["PodiumModel"] = pod.predict_proba(te[EARLY])[:, 1]
    te["PodiumRunning"] = [tab.get((min(r, 20), b), 0.0) for r, b in zip(te["Rank"], bucket(te))]
    te["WinRaw"] = np.where(te["Rank"] <= 4, win.predict_proba(win_x(te))[:, 1], 0.0)
    te["PodiumP"], te["WinP"] = np.nan, 0.0
    for _, idx in te.groupby(K).groups.items():
        g = te.loc[idx]
        early = g["LapFrac"].iloc[0] <= EARLY_CUT
        te.loc[idx, "PodiumP"] = norm_to(g["PodiumModel" if early else "PodiumRunning"].to_numpy(), min(3, len(g)))
        top = g["Rank"] <= 4
        te.loc[g.index[top], "WinP"] = norm_to(g.loc[top, "WinRaw"].to_numpy(), 1)
    te["PodiumSource"] = np.where(te["LapFrac"] <= EARLY_CUT, "model", "running order")
    out.append(te)
    print(year, "done:", te["Race"].nunique(), "races")
pred = pd.concat(out, ignore_index=True)

In [ ]:
def top_hits(df, score, label, k):
    s = df.assign(_s=np.asarray(score)).sort_values(K + ["_s"], ascending=[True, True, True, False])
    return (s.groupby(K).head(k).groupby(K)[label].sum() / k).groupby(level=["Year", "Race"]).mean()

def compare(df, score, label, k, lo, hi):
    w = df[(df["LapFrac"] > lo) & (df["LapFrac"] <= hi)]
    sc = np.asarray(score)[((df["LapFrac"] > lo) & (df["LapFrac"] <= hi)).to_numpy()]
    m, b = top_hits(w, sc, label, k), top_hits(w, -w["Rank"], label, k)
    g = (m - b).to_numpy()
    rng = np.random.default_rng(0)
    boot = [g[rng.integers(0, len(g), len(g))].mean() for _ in range(2000)]
    return dict(Model=m.mean(), RunningOrder=b.mean(), Gain=g.mean(), CI_Lo=np.percentile(boot, 2.5), CI_Hi=np.percentile(boot, 97.5), Races=len(g))

bt_src = pred[pred["Year"] >= 2022]
rows = []
for name, score, label, k in (("Podium", bt_src["PodiumP"], "Podium", 3), ("Winner", bt_src["WinP"] + (bt_src["Rank"] == 1) * 1e-6, "Win", 1)):
    for lo, hi in ((0, 0.5), (0.5, 1.0), (0.7, 1.0), (0, 1.0)):
        rows.append(dict(Target=name, Window=f"{int(lo * 100)}-{int(hi * 100)}%", **compare(bt_src, score, label, k, lo, hi)))
backtest = pd.DataFrame(rows)
SHIP_PODIUM = bool(backtest.query("Target == 'Podium' and Window == '0-50%'")["CI_Lo"].iloc[0] > 0)
SHIP_WINNER = bool(backtest.query("Target == 'Winner' and Window == '70-100%'")["CI_Lo"].iloc[0] > 0)
backtest["Ship"] = np.where(backtest["Target"] == "Podium", SHIP_PODIUM, SHIP_WINNER)
print(backtest.round(4).to_string(index=False))
print("Podium ship rule (0-50% CI above zero):", SHIP_PODIUM, "| Winner ship rule (70-100% CI above zero):", SHIP_WINNER)

In [ ]:
keep = ["Year", "Race", "LapNumber", "Driver", "Rank", "PodiumP", "WinP", "PodiumSource"]
spark.createDataFrame(pred[keep]).write.mode("overwrite").saveAsTable("workspace.default.f1_live_predictions")
spark.createDataFrame(backtest).write.mode("overwrite").saveAsTable("workspace.default.f1_live_backtest")
print("Wrote", len(pred), "rows; seasons", sorted(pred["Year"].unique()))

In [ ]:
# final models on every season, for the laptop live mode (copied to the site data by phase_5_site_export)
import json

OUT = "/Volumes/workspace/default/f1_data/live_models"
os.makedirs(OUT, exist_ok=True)
pod, win = fit_podium(d), fit_win(d)
tab, _ = running_lookup(d)
pod.save_model(f"{OUT}/podium_xgb.json")
json.dump({"coef": win.coef_[0].tolist(), "intercept": float(win.intercept_[0])}, open(f"{OUT}/win_logit.json", "w"))
json.dump({f"{int(r)},{int(b)}": float(v) for (r, b), v in tab.items()}, open(f"{OUT}/running.json", "w"))
print("Saved live models to", OUT)